<a href="https://colab.research.google.com/github/essensha/cs2-anti-cheat-ml-research/blob/main/Real_Time_Production_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Stage 1: Backlog and Investigative Review

AThe anti-cheatnti-cheat system receives 12 million gameplay events daily. Traffic during peak hours is 5 times the daily average.

In [ ]:
import pandas as pd

file_path = "/srv/synthetic_anticheat_telemetry.csv"

# Reload and overwrite the old DataFrame
df = pd.read_csv(file_path, skiprows=1, index_col=False)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(5))

Shape: (12000, 11)
Columns: ['player_id', 'match_id', 'match_day', 'skill_proxy', 'actions_per_min', 'median_reaction_ms', 'timing_regularities', 'target_accuracy', 'repeated_action_ratio', 'synthetic_pattern', 'is_suspicious_simulated']


,player_id,match_id,match_day,skill_proxy,actions_per_min,median_reaction_ms,timing_regularities,target_accuracy,repeated_action_ratio,synthetic_pattern,is_suspicious_simulated
0,100000,204010,27,0.616,151.3,334.8,0.383,0.510,0.066,normal,0
1,100001,200988,18,0.333,143.9,290.8,0.240,0.366,0.026,normal,0
2,100002,204639,52,0.741,155.3,172.5,0.537,0.613,0.421,normal,0
3,100003,203584,31,0.328,127.1,249.7,0.255,0.309,0.146,normal,0
4,100004,204014,19,0.418,129.4,247.0,0.176,0.460,0.204,normal,0


In [ ]:
daily_events = 12_000_000
seconds_per_day = 24 * 60 * 60

average_eps = daily_events / seconds_per_day
peak_eps = average_eps * 5

print(f"Peak events/sec: {peak_eps:.1f}")

Peak events/sec: 694.4


The peak thoughtput is 694.4 events/sec, truly 5 times the daily average. Riot should design event-processing system to best account for higher traffic to adjust to natural lifts during major beats which occur frequently due to their evolving gameplay change event calendar. This would also set reasonable expectations on opertation costs particularly with avoiding natural rising in investigative review

In [ ]:
import math

peak_eps = 694.4
worker_capacity = 100
buffer = 1.30

required_workers = math.ceil(
    peak_eps * buffer / worker_capacity
)

print("Required workers:", required_workers)

Required workers: 10


We round up because 9 workers provide only 900 events/second, below the buffered target of 902.72 events/second.

Let's quantify what happens if we keep only 10 workers during that spike.

In [ ]:
incoming_eps = 1388.9
workers = 10
capacity_per_worker = 100

processing_eps = workers * capacity_per_worker
backlog_per_second = incoming_eps - processing_eps

backlog_after_60_seconds = backlog_per_second * 60

print("Backlog after 60 seconds:", round(backlog_after_60_seconds))

Backlog after 60 seconds: 23334


Woah! We need to scale workers based on incoming event rates, processing throughput, and queue lag—not CPU utilization alone

In [ ]:
backlog = 23334
incoming_eps = 1389
processing_eps = 1900

net_drain_rate = processing_eps - incoming_eps
seconds_to_clear = backlog / net_drain_rate

print(round(seconds_to_clear, 1))

45.7


### Stage 2: Real-Time Feature Engineering

In [ ]:
import pandas as pd

events = pd.DataFrame({
    "player_id": ["A", "A", "A", "B", "B"],
    "second": [10, 30, 80, 20, 75],
    "suspicious": [1, 1, 1, 1, 0]
})

def count_recent(events, player, current_second):
    recent = events[
        (events["player_id"] == player)
        & (events["second"] > current_second - 60)
        & (events["second"] <= current_second)
    ]
    return recent["suspicious"].sum()

print("Player A:", count_recent(events, "A", 80))
print("Player B:", count_recent(events, "B", 80))

Player A: 2
Player B: 0


Retain both short-term and historical features. Comparing recent behavior against a player's historical baseline can reveal unusual changes without treating every skilled action as suspicious.

### Stage 2: Handling late events

- Rolling window (60 seconds): Determines which gameplay events belong in a feature calculation.
- Allowed lateness (90 seconds): Determines how long the pipeline accepts delayed events.

In [ ]:
event_time = 30
arrival_time = 100
window_end = 80

allowed_lateness = 90
window_size = 60

is_accepted = (
    arrival_time - event_time <= allowed_lateness
)

belongs_to_window = (
    window_end - window_size < event_time <= window_end
)

print("Accepted:", is_accepted)
print("Belongs to historical window:", belongs_to_window)

Accepted: True
Belongs to historical window: True


Accepted: True — the event arrived within the 90-second lateness allowance.



Belongs to historical window: True — its timestamp falls within the 60-second window ending at second 80.

In [ ]:
event_time = 30
arrival_time = 100
window_end = 80

allowed_lateness = 90
window_size = 60

is_accepted = (
    arrival_time - event_time <= allowed_lateness
)

belongs_to_window = (
    window_end - window_size < event_time <= window_end
)

print("Accepted:", is_accepted)
print("Belongs to historical window:", belongs_to_window)

Accepted: True
Belongs to historical window: True


### Stage 3: Real-Time Model Evaluation

In [ ]:
tp, fp, fn, tn = 180, 60, 120, 9640

precision = tp / (tp + fp)
recall = tp / (tp + fn)
fpr = fp / (fp + tn)

print(f"Precision: {precision:.1%}")
print(f"Recall: {recall:.1%}")
print(f"FPR: {fpr:.2%}")

Precision: 75.0%
Recall: 60.0%
FPR: 0.62%


I would establish guardrails for both precision and FPR, including segment-level FPR. A low overall FPR can still affect many legitimate players at Riot's scale.

Idempotency means processing the same event multiple times produces the same result as processing it once. Helps to avoid duplicates

In [ ]:
processed_ids = set()
suspicious_count = 0

def process_event(event_id, suspicious):
    global suspicious_count

    if event_id in processed_ids:
        return  # Already processed

    suspicious_count += suspicious
    processed_ids.add(event_id)